In [135]:
import pandas as pd
import numpy as np

In [136]:
ratings = pd.read_csv("../data/ratings.csv")
movies = pd.read_csv("../data/movies.csv")
tags = pd.read_csv("../data/tags.csv")
links = pd.read_csv("../data/links.csv")
ratings.drop(columns=['timestamp'], inplace=True)

In [137]:
#사용자 몇 명인지
ratings["userId"].nunique()

610

In [138]:
#영화 몇 편인지
ratings["movieId"].nunique()
#movies.shape에선 9742편 나옴 -> 18편은 아무도 평가X

9724

In [139]:
ratings = ratings.drop(columns=["timestamp"])


KeyError: "['timestamp'] not found in axis"

In [ ]:
movie_data = pd.merge(
    ratings, 
    movies, 
    on="movieId"
    )

In [ ]:
movie_data = movie_data[
    ["userId", "movieId", "title", "rating"]
]

In [ ]:
movie_data.shape

(100836, 4)

In [ ]:
movie_data.describe()

,userId,movieId,rating
count,100836.000000,100836.000000,100836.000000
mean,326.127564,19435.295718,3.501557
std,182.618491,35530.987199,1.042529
min,1.000000,1.000000,0.500000
25%,177.000000,1199.000000,3.000000
50%,325.000000,2991.000000,3.500000
75%,477.000000,8122.000000,4.000000
max,610.000000,193609.000000,5.000000


In [150]:
movie_data.head()

,userId,movieId,title,rating
0,1,1,Toy Story (1995),4.0
1,1,3,Grumpier Old Men (1995),4.0
2,1,6,Heat (1995),4.0
3,1,47,Seven (a.k.a. Se7en) (1995),5.0
4,1,50,"Usual Suspects, The (1995)",5.0


In [156]:
user_movie_matrix = train_movie_data.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

In [157]:
#처음 데이터에서 영화 9742편 중 아무도 평가하지 않은 영화는 제외
user_movie_matrix.shape


(610, 8960)

In [158]:
user_movie_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",(500) Days of Summer (2009),*batteries not included (1987),...All the Marbles (1981),...And Justice for All (1979),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [159]:
#Cosine은 NaN값을 처리할 수 없기 때문에, NaN값을 0으로 채워줌   
user_movie_matrix_filled = user_movie_matrix.fillna(0)


In [160]:
#Cosine 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix_filled)


In [161]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

In [162]:
user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.000000,0.077231,0.125869,0.099466,0.125327,0.130960,0.110490,0.048835,0.009627,...,0.055742,0.106239,0.180556,0.067582,0.128423,0.136302,0.232624,0.240208,0.074217,0.134951
2,0.000000,1.000000,0.000000,0.000000,0.021936,0.033434,0.027961,0.000000,0.000000,0.025923,...,0.213082,0.022625,0.015860,0.000000,0.000000,0.021944,0.000000,0.052219,0.037293,0.082265
3,0.077231,0.000000,1.000000,0.000000,0.006459,0.005063,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.002398,0.029349,0.000000,0.000000,0.013645,0.024847,0.027253,0.000000,0.034360
4,0.125869,0.000000,0.000000,1.000000,0.127775,0.100507,0.069582,0.054636,0.014229,0.024807,...,0.074343,0.103785,0.234084,0.050947,0.077667,0.144782,0.112395,0.117669,0.036205,0.076583
5,0.099466,0.021936,0.006459,0.127775,1.000000,0.261289,0.089896,0.248282,0.000000,0.037985,...,0.038918,0.292236,0.111600,0.190331,0.106604,0.078840,0.110768,0.110498,0.154324,0.058416


In [163]:
type(user_similarity)

numpy.ndarray

In [164]:
user_similarity.shape

(610, 610)

In [165]:
#user1과 유사한 사용자 찾기
similar_users = (
    user_similarity_df.loc[1]
    .drop(1)
    .sort_values(ascending=False)
)

similar_users.head(10)

userId
368    0.301938
266    0.297092
57     0.296645
313    0.290049
330    0.287225
91     0.279521
45     0.271869
480    0.271086
597    0.262294
217    0.257501
Name: 1, dtype: float64

In [ ]:
#266번 사용자가 평가한 영화 중 평점이 4.0 이상인 영화만 추출
# ratings_4 = ratings[
#     (ratings["userId"] == 266) 
#     & (ratings["rating"] >= 4.0)
# ]

In [166]:
#상위 10명의 유사한 사용자 추출
top_similar_users = similar_users.head(10).index

top_similar_users

Index([368, 266, 57, 313, 330, 91, 45, 480, 597, 217], dtype='int64', name='userId')

In [167]:
#위 10명의 평점 가져오기
similar_ratings = ratings [
    ratings['userId'].isin(top_similar_users)
]

similar_ratings.head()

,userId,movieId,rating
6477,45,1,4.0
6478,45,5,3.0
6479,45,6,4.0
6480,45,7,3.0
6481,45,11,3.0


In [168]:
#(실험용) 4.0이상 필터 만들기
similar_ratings_4 = similar_ratings[
    similar_ratings['rating'] >= 4.0
]

similar_ratings_4.head()

# similar_ratings_35 = similar_ratings[
#     similar_ratings['rating'] >= 3.5
# ]

# similar_ratings_35.head()

,userId,movieId,rating
6477,45,1,4.0
6479,45,6,4.0
6482,45,19,4.5
6483,45,21,4.0
6484,45,32,4.5


In [169]:
#user1이 본 영화 목록 (movieId만 추출)
watched_movies = ratings[
    ratings['userId'] == 1
]['movieId']

watched_movies.head()

#user1이 본 영화 개수 확인
#len(watched_movies)

0     1
1     3
2     6
3    47
4    50
Name: movieId, dtype: int64

In [170]:
candidate_movies = similar_ratings[
    ~similar_ratings['movieId'].isin(watched_movies)
]   

candidate_movies.head()

,userId,movieId,rating
6478,45,5,3.0
6480,45,7,3.0
6481,45,11,3.0
6482,45,19,4.5
6483,45,21,4.0


In [171]:
candidate_movies.groupby('movieId').size().head(10)

movieId
2     4
5     1
7     2
9     1
10    7
11    3
12    1
16    5
17    2
19    4
dtype: int64

In [172]:
#user1기준으로 유사도 측정
candidate_movies = candidate_movies.copy()

candidate_movies['similarity'] = candidate_movies['userId'].map(similar_users)

candidate_movies.head()

,userId,movieId,rating,similarity
6478,45,5,3.0,0.271869
6480,45,7,3.0,0.271869
6481,45,11,3.0,0.271869
6482,45,19,4.5,0.271869
6483,45,21,4.0,0.271869


In [173]:
#가중평균 계산함수 만들기
def weighted_average(group):
    return(
        (group['rating'] * group['similarity']).sum() 
        / group['similarity'].sum()
    )


In [180]:
# 추천 함수
#각각에 들어가는 기술들 정리ex)매핑, 코사인 유사도, 가중평균 등등
def recommend_movies(user_id, top_n=10, threshold=0):

    # 유사한 사용자 찾기
    similar_users = (
        user_similarity_df.loc[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
    )

    # 상위 10명의 유사한 사용자
    top_similar_users = similar_users.head(10).index

    # 유사한 사용자들의 평점 가져오기
    similar_ratings = train_movie_data[
        train_movie_data['userId'].isin(top_similar_users)
    ]

    # Threshold 설정
    similar_ratings = similar_ratings[
    similar_ratings['rating'] >= threshold
    ]

    # 사용자가 이미 본 영화
    watched_movies = train_movie_data[
        train_movie_data['userId'] == user_id
    ]['movieId']

    # 이미 본 영화 제거
    candidate_movies = similar_ratings[
        ~similar_ratings['movieId'].isin(watched_movies)
    ].copy()

    # 유사도 추가
    candidate_movies['similarity'] = (
        candidate_movies['userId'].map(similar_users)
    )

    # 예상 평점 계산
    predicted_ratings = (
        candidate_movies
        .groupby('movieId')
        .apply(weighted_average)
        .sort_values(ascending=False)
    )

    # 추천 결과 생성
    recommendations = (
        predicted_ratings
        .reset_index(name='predicted_rating')
        .merge(movies, on='movieId')
        .sort_values(
            by='predicted_rating',
            ascending=False
        )
        [['title', 'genres', 'predicted_rating']]
    )

    return recommendations.head(top_n)

In [181]:
#training set과 test set 나누기
from sklearn.model_selection import train_test_split

train_list = []
test_list = []

for user_id, group in movie_data.groupby('userId'):

    train, test = train_test_split(
        group, 
        test_size=0.2, 
        random_state=42
    )

    train_list.append(train)
    test_list.append(test)

train_movie_data = pd.concat(train_list)
test_movie_data = pd.concat(test_list)

In [182]:
print(train_movie_data.shape)
print(test_movie_data.shape)

(80419, 4)
(20417, 4)


In [183]:
print(train_movie_data['userId'].nunique())
print(test_movie_data['userId'].nunique())

610
610


In [184]:
#user1이 실제로 좋아한 영화가 뭐였는지 test set에서 확인
#여기서 Threshold는 4.0으로 설정
user_id = 1
threshold = 4.0

actual_movies = test_movie_data[
    (test_movie_data["userId"] == user_id) &
    (test_movie_data["rating"] >= threshold)
]

actual_movies[["title", "rating"]]

,title,rating
219,Gladiator (2000),5.0
66,"Abyss, The (1989)",4.0
9,Canadian Bacon (1995),5.0
15,Star Wars: Episode IV - A New Hope (1977),5.0
201,"Messenger: The Story of Joan of Arc, The (1999)",5.0
25,"Fugitive, The (1993)",5.0
197,Being John Malkovich (1999),4.0
154,Romancing the Stone (1984),4.0
126,"Negotiator, The (1998)",5.0
216,Ladyhawke (1985),4.0
